In [1]:
import requests
import pandas as pd
import sqlite3

# ==========================================
# 1. ETL AŞAMASI (Veriyi Çek, Temizle, Yükle)
# ==========================================
url = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&order=market_cap_desc&per_page=50&page=1&sparkline=false"
ham_veri = requests.get(url).json()

df = pd.DataFrame(ham_veri)
secilen_kolonlar = ['symbol', 'name', 'current_price', 'total_volume', 'price_change_percentage_24h']
df = df[secilen_kolonlar]

df.rename(columns={
    'symbol': 'sembol', 
    'name': 'coin_adi', 
    'current_price': 'fiyat_usd', 
    'total_volume': 'islem_hacmi_usd', 
    'price_change_percentage_24h': 'degisim_24s_yuzde'
}, inplace=True)
df.dropna(subset=['fiyat_usd', 'islem_hacmi_usd'], inplace=True)

# Veritabanına kaydet (Eski verinin üstüne yazar ki tabloda mükerrerlik olmasın)
conn = sqlite3.connect('kripto_piyasa.db')
df.to_sql('gunluk_piyasa_ozeti', conn, if_exists='replace', index=False)

# ==========================================
# 2. İLERİ SEVİYE SQL AŞAMASI (Analiz)
# ==========================================
karmaşık_sorgu = """
WITH PiyasaAnalizi AS (
    SELECT 
        sembol,
        coin_adi,
        fiyat_usd,
        islem_hacmi_usd,
        degisim_24s_yuzde,
        RANK() OVER (ORDER BY degisim_24s_yuzde DESC) as kazandirma_sirasi,
        RANK() OVER (ORDER BY islem_hacmi_usd DESC) as hacim_sirasi
    FROM gunluk_piyasa_ozeti
)
SELECT 
        kazandirma_sirasi,
        sembol,
        coin_adi,
        fiyat_usd,
        degisim_24s_yuzde
FROM PiyasaAnalizi
WHERE kazandirma_sirasi <= 5;
"""

# Sorguyu çalıştır ve sonucu al
df_analiz = pd.read_sql_query(karmaşık_sorgu, conn)
conn.close()

# Çıkan içgörüyü (insight) ekranda göster
df_analiz

,kazandirma_sirasi,sembol,coin_adi,fiyat_usd,degisim_24s_yuzde
0,1,wlfi,World Liberty Financial,0.056948,8.63157
1,2,xmr,Monero,531.840000,3.57998
2,3,shib,Shiba Inu,0.000005,3.21769
3,4,uni,Uniswap,6.350000,2.73192
4,5,cro,Cronos,0.058248,2.49166
